# Data Ingestion, Streaming & Sinks (`read_*`, `scan_*`, `write_*`)

`pytae` provides fast, dedicated readers and writers powered by Polars' multithreaded I/O core, complete with automatic compression handling, schema inference, and lazy query optimization.

- **Eager Readers**: `pt.read_parquet()`, `pt.read_csv()`, `pt.read_txt()`, `pt.read_jsonl()`, `pt.read_sas()`
- **Lazy Scanners**: `pt.scan_parquet()`, `pt.scan_csv()` for out-of-core streaming and query pushdowns
- **Writers**: `df.write_parquet()`, `df.write_csv()`, `df.write_jsonl()`
- **Sample Datasets**: Instant access to `penguins`, `diamonds`, `tips`, `flights`, `titanic` via `pt.sample()`

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt
import tempfile
from pathlib import Path

## 1. Built-in Sample Datasets (`pt.sample`)
Load sample datasets directly without downloading external files:

In [2]:
penguins = pt.sample("penguins")
display(penguins.head(4))
print("Available sample datasets:", list(pt.sample_data.keys()))

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Torgersen""",39.1,18.7,181.0,3750.0,"""Male"""
"""Adelie""","""Torgersen""",39.5,17.4,186.0,3800.0,"""Female"""
"""Adelie""","""Torgersen""",40.3,18.0,195.0,3250.0,"""Female"""
"""Adelie""","""Torgersen""",null,null,null,null,null


Available sample datasets: ['anagrams', 'anscombe', 'attention', 'brain_networks', 'car_crashes', 'diamonds', 'dots', 'dowjones', 'exercise', 'flights', 'fmri', 'geyser', 'glue', 'healthexp', 'iris', 'mpg', 'penguins', 'planets', 'seaice', 'taxis', 'tips', 'titanic']


## 2. Writing & Reading Parquet and CSV Files
Parquet provides columnar compression, fast reads, and preserved data types:

In [3]:
with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    pq_file = tmp_path / "penguins.parquet"
    csv_file = tmp_path / "penguins.csv"
    
    # Write to Parquet and CSV
    penguins.write_parquet(pq_file)
    penguins.write_csv(csv_file)
    
    # Read back eagerly
    df_pq = pt.read_parquet(pq_file)
    df_csv = pt.read_csv(csv_file)
    
    print(f"Parquet shape: {df_pq.shape}, CSV shape: {df_csv.shape}")
    display(df_pq.head(3))

Parquet shape: (344, 7), CSV shape: (344, 7)


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Torgersen""",39.1,18.7,181.0,3750.0,"""Male"""
"""Adelie""","""Torgersen""",39.5,17.4,186.0,3800.0,"""Female"""
"""Adelie""","""Torgersen""",40.3,18.0,195.0,3250.0,"""Female"""


## 3. Lazy Scanning & Query Pushdowns (`pt.scan_parquet`)
`pt.scan_parquet()` returns a lazy DataFrame (`pl.LazyFrame`). Filters and column selections push down directly into the Parquet reader:
Rows that do not match the filter are skipped in row groups before ever touching RAM!

In [4]:
with tempfile.TemporaryDirectory() as tmp_dir:
    pq_file = Path(tmp_dir) / "data.parquet"
    penguins.write_parquet(pq_file)
    
    # Scan lazily
    lazy_df = pt.scan_parquet(pq_file)
    print("Is lazy frame:", lazy_df.is_lazy)
    
    # Build query plan: filter & select
    query = (
        lazy_df
        .filter("body_mass_g > 4500 and island == 'Biscoe'")
        .select("species", "body_mass_g")
    )
    
    # Collect query results into eager DataFrame
    eager_res = query.collect()
    print("Collected shape:", eager_res.shape)
    display(eager_res.head(5))

Is lazy frame: True
Collected shape: (109, 2)


species,body_mass_g
str,f64
"""Adelie""",4725.0
"""Adelie""",4775.0
"""Adelie""",4600.0
"""Gentoo""",5700.0
"""Gentoo""",5700.0


## 4. Clipboard Operations
Quickly copy and paste data directly between Python and spreadsheets:

In [5]:
# df.to_clipboard() copies tab-delimited records to the OS clipboard
small_df = penguins.select("species", "body_mass_g").head(3)
# small_df.to_clipboard()  # Copies to OS clipboard
display(small_df)

species,body_mass_g
str,f64
"""Adelie""",3750.0
"""Adelie""",3800.0
"""Adelie""",3250.0
